# Causal Counterfactual Playbook

This companion notebook demonstrates how to parameterise counterfactual scenarios using the causal toolkit.

In [ ]:
from entropy_news.research.causal import (
    CausalPanelConfig,
    assemble_causal_panel,
    build_propensity_features,
    difference_in_differences,
    two_stage_least_squares,
    synthetic_control,
    build_summary_table,
    PolicyScenario,
    prepare_counterfactual_series,
    format_policy_narrative,
)
import pandas as pd

entropy = pd.DataFrame(
    {
        "unit": ["treated", "control_a", "control_b"] * 3,
        "time": [0, 0, 0, 1, 1, 1, 2, 2, 2],
        "treatment": [1, 0, 0, 1, 0, 0, 1, 0, 0],
        "post": [0, 0, 0, 1, 1, 1, 1, 1, 1],
        "instrument": [0.4, 0.2, 0.2, 1.2, 0.3, 0.3, 1.2, 0.4, 0.4],
    }
)
market = pd.DataFrame(
    {
        "unit": ["treated", "control_a", "control_b"] * 3,
        "time": [0, 0, 0, 1, 1, 1, 2, 2, 2],
        "outcome": [10.0, 10.0, 10.2, 14.0, 10.7, 11.05, 15.0, 11.4, 11.9],
        "volatility": [0.6, 0.5, 0.5, 0.75, 0.7, 0.7, 0.9, 0.85, 0.85],
    }
)
config = CausalPanelConfig(
    unit_col="unit",
    time_col="time",
    outcome_col="outcome",
    treatment_col="treatment",
    post_treatment_col="post",
    covariate_cols=("volatility",),
    instrument_cols=("instrument",),
)
panel = assemble_causal_panel(entropy, market, config)
propensity = build_propensity_features(panel, config, window=2)
did = difference_in_differences(panel, config)
iv = two_stage_least_squares(panel, config)
sc = synthetic_control(panel, config, treated_unit="treated", donor_units=["control_a", "control_b"])
summary = build_summary_table(did, iv_result=iv, sc_result=sc)
scenario = PolicyScenario(
    name="Liquidity Injection",
    description="Add liquidity during crisis weeks",
    target_group="primary dealers",
)
series = prepare_counterfactual_series(sc)
print(summary)
print(series.head())
print(format_policy_narrative(scenario, did))
